My dataset contains the information about cancer patients. 
Features: 
sex (1 - male, 2 - female)
Immun - immunity phenotype
CNS - CNS status
Mediastinum - mediastinum status
Zytogen - genetic error (0 - no, error code otherwise)
Region - region code
Geb_month - month of birth
Diag_motnh - month of revealing the diagnosis
syndrome - Down syndrome
Leuc - leucocyte amount in blood
Leber - size of liver
Milz - size of spleen
height, weight, age.

The target feature is a type of treatment and it's coded as "100" - first type, "200" - second type, "0" - doesn't matter which is the best in this case.

The method for classifying the observations is decision tree classifier. It will be run on full and reduced data and in different cases we'll try to use different parameters of the model

Firstly, download the data, check the shape.

In [41]:
from __future__ import division
import numpy as np
from sklearn.cross_validation import train_test_split
from sklearn import tree

In [24]:
data = np.genfromtxt('D:\Study\HSE\Books\kuznetsov\BIGHW\\2008_100_200.csv', delimiter=";", skip_header=1)
print(data.shape)

(805L, 17L)


Deleting the rows with missing data and the rows with target classified as "0" which means that the best type of treatment for this person is unknown (nor "100" neither "200"). Tracing changes in data size.

In [30]:
data = data[~np.isnan(data).any(axis=1)]
data = data[np.logical_not(data[:, 16] == 0)]
print(data.shape)

(760L, 17L)


Now we see that we had to get rid of 45 observations. 
Next step: creating the model, splitting the data for fitting, cross-validation.

In [31]:
clf = tree.DecisionTreeClassifier(random_state=0)
data_train, data_test = train_test_split(data, test_size=0.5, random_state=0)
target = [x[16] for x in data_train]
train = data_train[:, 0:16]
fitted_tree = clf.fit(train, target)
test = data_test[:, 0:16]

In [32]:
predicted_probs = clf.predict_proba(test)
predicted_probs = predicted_probs.astype(int)
test_to_compare = np.array(data_test[:, 16])

Now we can merge the result with the column of real class labels to see what happens (first 100 rows).

In [33]:
compare = np.column_stack((predicted_probs, test_to_compare))
print(compare[:100, :])

[[   1.    0.  200.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  200.]
 [   0.    1.  200.]
 [   1.    0.  200.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  200.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    1.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    1.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    1.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    1.  100.]
 [   1.    0.  100.]
 [   0.    1.  200.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  200.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    1.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.

In [ ]:
Computing the measures of fitting:

In [34]:
TP = 0
TN = 0
FP = 0
FN = 0
#class 100 is +, 200 is -
for row in compare:
    if (row[0] == 1) & (row[2] == 100):
        TP += 1
    if (row[1] == 1) & (row[2] == 200):
        TN += 1
    if (row[1] == 1) & (row[2] == 100):
        FN += 1
    if (row[0] == 1) & (row[2] == 200):
        FP += 1

TPR_recall = TP/(TP+FN)
TNR = TN/(FP+TN)
NPV = TN/(TN+FN)
FPR = FP/(FP+TN)
FDR_precision = float(FP/(FP+TP))
Accuracy = (TP+TN)/(TP+TN+FP+FN)
summary = {"TP": TP, "TN": TN, "FP": FP, "FN": FN,
           "TPR_recall": TPR_recall, "TNR": TNR, "NPV": NPV, "FPR": FPR,
           "FDR_precision": FDR_precision, "Accuracy": Accuracy}
print(summary)

{'TN': 27, 'FP': 28, 'FN': 34, 'FPR': 0.509090909090909, 'TNR': 0.4909090909090909, 'FDR_precision': 0.0877742946708464, 'NPV': 0.4426229508196721, 'TPR_recall': 0.8953846153846153, 'TP': 291, 'Accuracy': 0.8368421052631579}


As we see, accuracy is quite high (0.83), but we can play with classifier parameters and with attributes in the data.

For example we can easily delete such features as ID, Geb_mon, CNS, syndrome and Mediastine (ID and Geb_mon don't play any role and the rest features have the same values everywhere). Hypothetically, region also doesn't have a great influence.

In [35]:
data = np.delete(data, [0, 3, 4, 10, 11, 14], 1)

In [40]:
clf = tree.DecisionTreeClassifier(max_features=3, max_depth=5, random_state=0)
data_train, data_test = train_test_split(data, test_size=0.5, random_state=0)
target = [x[10] for x in data_train]
train = data_train[:, 0:10]
fitted_tree = clf.fit(train, target)
test = data_test[:, 0:10]

predicted_probs = clf.predict_proba(test)
predicted_probs = predicted_probs.astype(int)
test_to_compare = np.array(data_test[:, 10])
print(test_to_compare.shape)

compare = np.column_stack((predicted_probs, test_to_compare))
print(compare[:100, :])

TP = 0
TN = 0
FP = 0
FN = 0
#class 100 is +, 200 is -
for row in compare:
    if (row[0] == 1) & (row[2] == 100):
        TP += 1
    if (row[1] == 1) & (row[2] == 200):
        TN += 1
    if (row[1] == 1) & (row[2] == 100):
        FN += 1
    if (row[0] == 1) & (row[2] == 200):
        FP += 1

TPR_recall = TP/(TP+FN)
TNR = TN/(FP+TN)
NPV = TN/(TN+FN)
FPR = FP/(FP+TN)
FDR_precision = FP/(FP+TP)
Accuracy = (TP+TN)/(TP+TN+FP+FN)
summary = {"TP": TP, "TN": TN, "FP": FP, "FN": FN,
           "TPR_recall": TPR_recall, "TNR": TNR, "NPV": NPV, "FPR": FPR,
           "FDR_precision": FDR_precision, "Accuracy": Accuracy}
print(summary)

(380L,)
[[   1.    0.  200.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    0.  100.]
 [   0.    0.  100.]
 [   1.    0.  200.]
 [   0.    1.  200.]
 [   0.    0.  200.]
 [   1.    0.  100.]
 [   0.    0.  100.]
 [   1.    0.  100.]
 [   0.    0.  100.]
 [   1.    0.  100.]
 [   0.    0.  200.]
 [   0.    0.  100.]
 [   1.    0.  100.]
 [   0.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    1.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  200.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    0.  200.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   1.    0.  100.]
 [   0.    1.  100.]
 [   0.    0.  100.]
 [   1.    0.  100.]
 [   

Now the accuracy is a bit higher (0.88), probably due to restricting parameters of fitting (max_features, max_depth) which can help the model avoid overfitting.